In [6]:
import pandas as pd

# Load your metadata
df = pd.read_csv('./dataset_nodule21/cxr_images/proccessed_data/metadata.csv')

# Extract patient ID from image name
df['patient_id'] = df['img_name'].str.extract(r'(n\d+)')

# Check how many unique images per patient
images_per_patient = df.groupby('patient_id')['img_name'].nunique()

print("="*50)
print("IMAGES PER PATIENT DISTRIBUTION:")
print("="*50)
print(images_per_patient.value_counts())
print(f"\nTotal unique patients: {df['patient_id'].nunique()}")
print(f"Total unique images: {df['img_name'].nunique()}")

# If any patient has more than 1 image, show them
multi_image_patients = images_per_patient[images_per_patient > 1]
if len(multi_image_patients) > 0:
    print(f"\n⚠️ WARNING: {len(multi_image_patients)} patients have multiple images:")
    print(multi_image_patients)
else:
    print("\n✓ Each patient has exactly one image.")
    print("✓ Image-level split = Patient-level split.")
    print("✓ Safe to claim patient-level splitting in your paper.")

# Load all three split CSVs
train_df = pd.read_csv('./dataset_nodule21/cxr_images/proccessed_data/split_data/train/metadata_train.csv')
val_df = pd.read_csv('./dataset_nodule21/cxr_images/proccessed_data/split_data/val/metadata_val.csv')
test_df = pd.read_csv('./dataset_nodule21/cxr_images/proccessed_data/split_data/test/metadata_test.csv')

# Extract patient IDs from each split
train_df['patient_id'] = train_df['img_name'].str.extract(r'(n\d+)')
val_df['patient_id'] = val_df['img_name'].str.extract(r'(n\d+)')
test_df['patient_id'] = test_df['img_name'].str.extract(r'(n\d+)')

# Get unique patient IDs per split
train_patients = set(train_df['patient_id'].dropna())
val_patients = set(val_df['patient_id'].dropna())
test_patients = set(test_df['patient_id'].dropna())

# Check overlaps at patient level
print("="*50)
print("PATIENT-LEVEL OVERLAP CHECK:")
print("="*50)
print(f"Train patients: {len(train_patients)}")
print(f"Val patients: {len(val_patients)}")
print(f"Test patients: {len(test_patients)}")

print(f"\nTrain-Val patient overlap: {len(train_patients & val_patients)} (should be 0)")
print(f"Train-Test patient overlap: {len(train_patients & test_patients)} (should be 0)")
print(f"Val-Test patient overlap: {len(val_patients & test_patients)} (should be 0)")

if len(train_patients & val_patients) == 0 and len(train_patients & test_patients) == 0 and len(val_patients & test_patients) == 0:
    print("\n✓ No patient-level overlap detected!")
    print("✓ Safe to claim patient-level splitting.")
else:
    print("\n WARNING: Patient appears in multiple splits!")
    print(" This may lead to data leakage in your evaluation.")

IMAGES PER PATIENT DISTRIBUTION:
img_name
1    1134
Name: count, dtype: int64

Total unique patients: 1134
Total unique images: 4882

✓ Each patient has exactly one image.
✓ Image-level split = Patient-level split.
✓ Safe to claim patient-level splitting in your paper.
PATIENT-LEVEL OVERLAP CHECK:
Train patients: 794
Val patients: 175
Test patients: 165

Train-Val patient overlap: 0 (should be 0)
Train-Test patient overlap: 0 (should be 0)
Val-Test patient overlap: 0 (should be 0)

✓ No patient-level overlap detected!
✓ Safe to claim patient-level splitting.


In [8]:
import pandas as pd

# Load all three split CSVs
train_df = pd.read_csv('./dataset_nodule21/cxr_images/proccessed_data/split_data/train/metadata_train.csv')
val_df = pd.read_csv('./dataset_nodule21/cxr_images/proccessed_data/split_data/val/metadata_val.csv')
test_df = pd.read_csv('./dataset_nodule21/cxr_images/proccessed_data/split_data/test/metadata_test.csv')

# Extract patient IDs from both naming conventions
def extract_patient_id(img_name):
    # Remove augmentation suffix first
    base = img_name.replace('.mha', '')
    base = base.split('_aug')[0]
    return base

train_df['patient_id'] = train_df['img_name'].apply(extract_patient_id)
val_df['patient_id'] = val_df['img_name'].apply(extract_patient_id)
test_df['patient_id'] = test_df['img_name'].apply(extract_patient_id)

# Get unique patient IDs per split
train_patients = set(train_df['patient_id'].unique())
val_patients = set(val_df['patient_id'].unique())
test_patients = set(test_df['patient_id'].unique())

print("="*50)
print("COMPLETE PATIENT-LEVEL OVERLAP CHECK:")
print("="*50)
print(f"Train patients: {len(train_patients)}")
print(f"Val patients: {len(val_patients)}")
print(f"Test patients: {len(test_patients)}")
print(f"Total unique patients: {len(train_patients | val_patients | test_patients)}")

print(f"\nTrain-Val patient overlap: {len(train_patients & val_patients)} (should be 0)")
print(f"Train-Test patient overlap: {len(train_patients & test_patients)} (should be 0)")
print(f"Val-Test patient overlap: {len(val_patients & test_patients)} (should be 0)")

if len(train_patients & val_patients) == 0 and \
   len(train_patients & test_patients) == 0 and \
   len(val_patients & test_patients) == 0:
    print("\n✓ CONFIRMED: No patient-level overlap!")
    print("✓ Image-level split = Patient-level split.")
    print("✓ Safe to claim patient-level splitting in paper.")
else:
    print("\n⚠️ WARNING: Same patient appears in multiple splits!")
    
    overlap_tv = train_patients & val_patients
    overlap_tt = train_patients & test_patients
    overlap_vt = val_patients & test_patients
    
    if overlap_tv:
        print(f"Train-Val overlapping patients: {list(overlap_tv)[:5]}")
    if overlap_tt:
        print(f"Train-Test overlapping patients: {list(overlap_tt)[:5]}")
    if overlap_vt:
        print(f"Val-Test overlapping patients: {list(overlap_vt)[:5]}")

COMPLETE PATIENT-LEVEL OVERLAP CHECK:
Train patients: 3417
Val patients: 732
Test patients: 733
Total unique patients: 4882

Train-Val patient overlap: 0 (should be 0)
Train-Test patient overlap: 0 (should be 0)
Val-Test patient overlap: 0 (should be 0)

✓ CONFIRMED: No patient-level overlap!
✓ Image-level split = Patient-level split.
✓ Safe to claim patient-level splitting in paper.
